# Multiscale links (cross-pyramid-level edges)

**Geometry type:** `graph`

This notebook is a deep dive into the **links layout**: how graph edges, cross-chunk links, cross-level links and their attributes are organised within and across pyramid levels, and how you can read each piece directly.

Topics:

1. The on-disk layout: `links/<delta>/<offsets>/` and `link_attributes/<name>/<delta>/<offsets>/`
2. Write a small graph and build a 3-level pyramid
3. Inspect the tree — the `<delta>` groups and `<offsets>` arrays per level
4. Read intra-level edges (`delta=0`), within and across chunks
5. Read cross-level edges (`delta=+1`) — fine → coarse drill-up
6. The mirrored `-1` family at the coarser level
7. Per-link attributes
8. Storage modes: `none` vs `implicit` vs `explicit`
9. Depth knob: `cross_level_depth=2`
10. Validate

In [ ]:
import numpy as np
import tempfile, os
from pathlib import Path

_tmpdir = tempfile.mkdtemp(prefix="zv_multiscale_")
STORE = os.path.join(_tmpdir, "graph.zarrvectors")
print("Store:", STORE)

## 1 · The on-disk layout

Connectivity is **one family** per resolution level.  Every link array sits under two path segments:

```
<level>/links/<delta>/<offsets>/                     one rank-D vlen array over the chunk grid
<level>/link_attributes/<name>/<delta>/<offsets>/    mirrors it cell for cell, row for row
```

`<delta>` says how many pyramid levels a record spans:

| Segment | Meaning |
|---------|---------|
| `0`     | intra-level edges |
| `+1`    | edges from this level to `this_level + 1` (one step coarser) |
| `-1`    | edges from this level to `this_level - 1` (one step finer) |
| `+N` / `-N` | jumps of N levels |

`<offsets>` says where the *other* endpoint sits relative to the record's **source chunk** — the array cell that holds the record:

- `0.0.0` — both endpoints in the source chunk (an intra-chunk link);
- `0.0.+1` — the other endpoint is one chunk along the last axis;
- and so on.  A cross-chunk link is simply a link whose offsets are not all zero; there is no separate cross-chunk family.

Each record row holds chunk-local vertex indices: column 0 is local to the source chunk, column 1 is local to chunk `source + offset` (at level `this_level + delta`).

Use the path helpers in `zarr_vectors.core.paths` to compose paths — never hard-code the `<delta>` or `<offsets>` formatting yourself.

In [ ]:
from zarr_vectors.core.paths import (
    format_delta, parse_delta,
    format_offsets, parse_offsets,
    links_group_path, links_path,
    link_attributes_group_path, link_attributes_path,
)

print("format_delta(0)  =", format_delta(0))
print("format_delta(+1) =", format_delta(1))
print("format_delta(-2) =", format_delta(-2))
print("parse_delta('+2') =", parse_delta("+2"))
print()
print("format_offsets([(0, 0, 1)]) =", format_offsets([(0, 0, 1)]))
print()
print("links_group_path(+1)                =", links_group_path(1))
print("links_path(0, [(0, 0, 0)])          =", links_path(0, [(0, 0, 0)]))
print("links_path(0, [(0, 0, 1)])          =", links_path(0, [(0, 0, 1)]))
print("links_path(-1, [(0, 0, 0)])         =", links_path(-1, [(0, 0, 0)]))
print("link_attributes_group_path('weight', +1)          =", link_attributes_group_path("weight", 1))
print("link_attributes_path('weight', 0, [(0, 0, 1)])    =", link_attributes_path("weight", 0, [(0, 0, 1)]))

## 2 · Write a small graph and build a 3-level pyramid

We use a small (500-node) chain graph in a 400 µm cube, chunked at 100 µm, so the pyramid produces a handful of metanodes per level — easy to eyeball.

`build_pyramid` defaults to `cross_level_depth=1` and `cross_level_storage="explicit"`; we pass them explicitly here and compare modes below.  `explicit` writes `+1` at the finer level and `-1` at the coarser level for every adjacent pair.

In [ ]:
from zarr_vectors.types.graphs import write_graph
from zarr_vectors.multiresolution.coarsen import build_pyramid
from zarr_vectors.constants import XLEVEL_EXPLICIT, XLEVEL_IMPLICIT, XLEVEL_NONE

rng = np.random.default_rng(0)
N = 500
positions = rng.uniform(0.0, 400.0, size=(N, 3)).astype(np.float32)
edges = np.stack([np.arange(N - 1), np.arange(1, N)], axis=1).astype(np.int64)
edge_weights = rng.uniform(0.1, 1.0, size=len(edges)).astype(np.float32)

write_graph(
    STORE,
    positions=positions,
    edges=edges,
    object_ids=np.zeros(N, dtype=np.int64),
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    edge_attributes={"weight": edge_weights},
)

summary = build_pyramid(
    STORE,
    factors=[(2.0, 1.0), (2.0, 1.0)],
    cross_level_depth=1,                  # ±1 between every adjacent pair
    cross_level_storage=XLEVEL_EXPLICIT,  # store both +1 and -1
)
print("Levels created:", summary["levels_created"])
print("Vertices per level:", [N] + [s["vertex_count"] for s in summary["level_specs"]])

## 3 · Walk the on-disk tree

Each level carries one `links/<delta>/` group per delta present, and each group holds one array per distinct offsets segment.

In [ ]:
from zarr_vectors.core.store import open_store, list_resolution_levels, get_resolution_level
from zarr_vectors.core.arrays import list_link_deltas, list_link_offsets

root = open_store(STORE)
levels = sorted(list_resolution_levels(root))
print(f"Pyramid levels: {levels}")
print()
for lvl in levels:
    lg = get_resolution_level(root, lvl)
    deltas = list_link_deltas(lg)
    segs = {format_delta(d): len(list_link_offsets(lg, d)) for d in deltas}
    print(f"  level {lvl}: links/<delta> = {[format_delta(d) for d in deltas]}"
          f"    offsets arrays per delta = {segs}")

Reading this:

- Level 0 has `0` (the graph's own edges) and `+1` (drill *up* to level 1), but no `-1` (nothing below).
- The middle level carries both `+1` and `-1`.
- The top level has `-1` but no `+1` (nothing above).
- `links/0` at level 0 has many offsets arrays: one for intra-chunk edges plus one per neighbour direction the chain crosses.

Level groups are named by their bare index (`0/`, `1/`, ...).  Peek at the directory:

In [ ]:
level0_links = Path(STORE) / "0" / "links"
print(f"contents of {level0_links}:")
for child in sorted(level0_links.iterdir()):
    print(" ", child.name)
print()
segments = sorted(p.name for p in (level0_links / "0").iterdir() if p.is_dir())
print(f"links/0 holds {len(segments)} offsets arrays, e.g. {segments[:4]}")

## 4 · Intra-level edges — `delta=0`

`read_links` reads a whole `<delta>` family and returns records as `((chunk_a, local_a), (chunk_b, local_b))` tuples, in the endpoint order they were written.  Records whose two chunks match are intra-chunk links; the rest cross a chunk boundary.  `include_intra=False` skips the all-zero offsets array.

`read_chunk_links` reads one cell of one offsets array — the raw per-chunk view.

In [ ]:
from zarr_vectors.core.arrays import read_links, read_chunk_links

lg0 = get_resolution_level(root, 0)
recs0 = read_links(lg0, delta=0)
n_intra = sum(1 for (ca, _), (cb, _) in recs0 if ca == cb)
print(f"level 0, delta=0: {len(recs0)} edges "
      f"({n_intra} intra-chunk, {len(recs0) - n_intra} cross-chunk)")
print(f"read_links(include_intra=False): {len(read_links(lg0, delta=0, include_intra=False))} records")
print("first 3 records:")
for (ca, la), (cb, lb) in recs0[:3]:
    print(f"  chunk={ca} local={la}  --  chunk={cb} local={lb}")

# One cell of the intra-chunk array: rows of (local_a, local_b).
ck = next(ca for (ca, _), (cb, _) in recs0 if ca == cb)
groups = read_chunk_links(lg0, ck, delta=0)
print(f"\nread_chunk_links(chunk {ck}, delta=0):", [g.tolist() for g in groups][:3])

## 5 · Cross-level edges — `delta=+1` (drill up)

Every fine vertex gets one edge to its coarse parent metanode.  The source endpoint is at the owning level; the target endpoint is at level `+1`.  The offsets segment is measured in the target level's chunk grid, so with the default (inherited) chunk shape every parent lands in offsets `0.0.0`.

The family is **directed**: endpoint order is preserved (fine → coarse), which `link_family_policy` reports.

In [ ]:
from zarr_vectors.core.arrays import link_family_policy

link_width, sid_ndim, directed, store_mode = link_family_policy(lg0, 1)
print(f"links/+1 policy: link_width={link_width} sid_ndim={sid_ndim} "
      f"directed={directed} store={store_mode!r}")
print("offsets arrays:", list_link_offsets(lg0, 1))

plus1 = read_links(lg0, delta=1)
print(f"\nlevel 0 has {len(plus1)} +1 edges (one per vertex)")
for (ca, la), (cb, lb) in plus1[:3]:
    print(f"  level-0 chunk={ca} local={la}  ->  level-1 chunk={cb} local={lb}")

## 6 · The mirrored `-1` family

Under `explicit` storage the coarser level carries the same edges pointing back down: `links/-1` at level 1 holds every `links/+1` record of level 0 with its endpoints swapped.

In [ ]:
lg1 = get_resolution_level(root, 1)
minus1 = read_links(lg1, delta=-1)
swapped = {(b, a) for a, b in plus1}
print(f"level 1 has {len(minus1)} -1 edges")
print("level-1 links/-1 == swapped level-0 links/+1:", set(minus1) == swapped)

## 7 · Per-link attributes

`link_attributes/<name>/<delta>/<offsets>/` mirrors `links/<delta>/<offsets>/` exactly — same cells, same per-cell row order — so attribute row `i` from `read_link_attributes` belongs to record `i` from `read_links`.  Nothing on disk ties a row to a record; the shared enumeration order is the contract.

`write_graph` wrote the `delta=0` `weight` attribute from `edge_attributes=`.  Below we also write a `delta=+1` attribute by hand.  Without a `partition` from the matching `write_links` call, the values must be in `read_links` order.  Writes check `len(values) == num_links`, so a misaligned write fails loudly instead of silently corrupting the parallel array.

In [ ]:
from zarr_vectors.core.arrays import (
    write_link_attributes,
    read_link_attributes,
    list_link_attribute_deltas,
)

w0 = read_link_attributes(lg0, "weight", delta=0)
print(f"weight at delta=0: {len(w0)} rows for {len(recs0)} records")

# Re-open for writing.
root_rw = open_store(STORE, mode="r+")
lg0_rw = get_resolution_level(root_rw, 0)

# One float per +1 record, in read_links order (a ramp, for illustration).
plus1_vals = np.linspace(0.0, 1.0, len(plus1), dtype=np.float32)
write_link_attributes(lg0_rw, "drill_weight", plus1_vals, num_links=len(plus1), delta=1)
back = read_link_attributes(lg0_rw, "drill_weight", delta=1)
print(f"wrote/read {len(back)} drill_weight rows at delta=+1; round-trip equal:",
      np.array_equal(back, plus1_vals))
print("deltas carrying drill_weight:", list_link_attribute_deltas(lg0_rw, "drill_weight"))

Length check (this *should* raise):

In [ ]:
from zarr_vectors.exceptions import ArrayError

try:
    bad = np.zeros(len(plus1) + 7, dtype=np.float32)
    write_link_attributes(lg0_rw, "drill_weight", bad, num_links=len(plus1), delta=1)
except ArrayError as e:
    print(f"ArrayError raised as expected:\n  {e}")

## 8 · Storage modes: `explicit` vs `implicit` vs `none`

Three modes control whether `-N` arrays are materialised at all:

| Mode | Writes `+N` at fine level? | Writes `-N` at coarse level? |
|------|----------------------------|------------------------------|
| `none`     | no  | no  |
| `implicit` | yes | no  |
| `explicit` | yes | yes |

`implicit` saves storage; the `-N` direction is reconstructed by reading the `+N` array at the target level and swapping endpoints.  `explicit` materialises both, paying disk for direct reads in both directions.

Let's rebuild against fresh stores to compare.

In [ ]:
import shutil

def _build_one(name, *, depth, storage):
    path = os.path.join(_tmpdir, f"{name}.zarrvectors")
    if os.path.exists(path):
        shutil.rmtree(path)
    write_graph(
        path,
        positions=positions,
        edges=edges,
        object_ids=np.zeros(N, dtype=np.int64),
        chunk_shape=(100.0, 100.0, 100.0),
        bin_shape=(25.0, 25.0, 25.0),
        edge_attributes={"weight": edge_weights},
    )
    build_pyramid(
        path, factors=[(2.0, 1.0), (2.0, 1.0)],
        cross_level_depth=depth, cross_level_storage=storage,
    )
    return path

def _scan(path):
    r = open_store(path)
    return [
        (lvl, [format_delta(d) for d in list_link_deltas(get_resolution_level(r, lvl))])
        for lvl in sorted(list_resolution_levels(r))
    ]

for mode in (XLEVEL_NONE, XLEVEL_IMPLICIT, XLEVEL_EXPLICIT):
    p = _build_one(f"graph_{mode}", depth=1, storage=mode)
    print(f"\n--- cross_level_storage = {mode!r} ---")
    for lvl, ld in _scan(p):
        print(f"  level {lvl}: links/<delta> = {ld}")

## 9 · Depth knob: `cross_level_depth=2`

`cross_level_depth` controls how far the cross-level emission reaches:

- `0` — disabled (same as `storage="none"`).
- `N` — materialise up to `±N` for every pair of levels it can reach.
- `-1` — walk *all* available pyramid levels.

At `depth=2` the writer composes the fine→parent map across two coarsening steps (`grandparent[i] = parent_at_L1[parent_at_L0[i]]`) so a single edge goes from a level-0 vertex straight to its level-2 metanode.

In [ ]:
p2 = _build_one("graph_depth2", depth=2, storage=XLEVEL_EXPLICIT)
print("depth=2, explicit:")
for lvl, ld in _scan(p2):
    print(f"  level {lvl}: links/<delta> = {ld}")

Expected (with a 3-level pyramid):

- Level 0 → `+1`, `+2`
- Level 1 → `-1`, `+1`
- Level 2 → `-1`, `-2`

Plus `0` at level 0 from the original `write_graph` call.

## 10 · Validate

The validator walks every level, including each `<delta>` group and `<offsets>` array under `links/`.

In [ ]:
from zarr_vectors.validate import validate

rv = validate(STORE, level=3)
print(rv.summary())

## Summary

| Concept | API |
|---------|-----|
| Pyramid with cross-level edges | `build_pyramid(path, factors=..., cross_level_depth=N, cross_level_storage="explicit")` |
| Compose paths | `links_group_path(delta)`, `links_path(delta, offsets)`, `link_attributes_group_path(name, delta)`, `link_attributes_path(name, delta, offsets)` |
| List what is on disk | `list_link_deltas(level)`, `list_link_offsets(level, delta)`, `list_link_attribute_deltas(level, name)` |
| Read a whole family | `read_links(level, delta=0)`, `read_links(level, delta=+1)` |
| Read one cell | `read_chunk_links(level, chunk, delta=..., offsets=...)` |
| Family policy | `link_family_policy(level, delta)` → `(link_width, sid_ndim, directed, store)` |
| Link attributes | `write_link_attributes(level, name, values, num_links=..., delta=...)` / `read_link_attributes(level, name, delta=...)` |

Endpoint convention recap:

- `links/<delta>/<offsets>/` cell `c` holds records whose source chunk is `c` at the owning level.
- Column 0 is local to chunk `c`; column 1 is local to chunk `c + offset` at level `this_level + delta`.
- `read_links` reassembles these into `((src_chunk, src_local), (tgt_chunk, tgt_local))` tuples.